# Smart Pest Detection — Model Training (Colab)

Trains **MobileNetV3-Large** to identify crop pests from photos, for the Smart Pest
Detection & Prediction System.

**19 classes** = 12 general insects (from the *Agricultural Pests Image Dataset*)
+ 7 Sri Lankan crop pests (from the **IP102** dataset):
brown_planthopper, rice_stem_borer, fall_armyworm, fruit_fly, thrips, mealybug, leafhopper.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

Outputs (downloaded at the end): `pest_model.keras`, `class_names.json`, `confusion_matrix.png`.
Copy the first two into the project's `backend/model/` folder (replacing the old ones).


## 1. Setup


In [ ]:
!pip -q install kagglehub
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
import json, os, shutil, glob
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Download both datasets

- **Dataset A** — Agricultural Pests Image Dataset (12 general classes).
- **Dataset B** — IP102 (large insect-pest benchmark) — we take only 7 crop-pest classes from it.

`kagglehub` downloads public datasets with no login. IP102 is large (a few GB), so this cell
can take several minutes.


In [ ]:
import kagglehub
path_general = kagglehub.dataset_download('vencerlanz09/agricultural-pests-image-dataset')
print('General dataset:', path_general)
path_ip102 = kagglehub.dataset_download('rtlmhjbn/ip02-dataset')
print('IP102 dataset:', path_ip102)

## 3. Set up a merged folder with one sub-folder per class

Everything gets copied into `/content/merged/<class_name>/`. `image_dataset_from_directory`
will then read the class names straight from these folder names.


In [ ]:
MERGED = '/content/merged'
if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
os.makedirs(MERGED)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def is_img(f):
    return os.path.splitext(f)[1].lower() in IMG_EXT

def copy_images(src_files, dst_dir, prefix, cap=None):
    os.makedirs(dst_dir, exist_ok=True)
    if cap is not None and len(src_files) > cap:
        rng = np.random.default_rng(42)
        src_files = list(rng.choice(src_files, size=cap, replace=False))
    n = 0
    for i, src in enumerate(src_files):
        try:
            shutil.copy(src, os.path.join(dst_dir, f'{prefix}_{i}{os.path.splitext(src)[1].lower()}'))
            n += 1
        except Exception:
            pass
    return n

## 4. Copy the 12 general classes (Dataset A)

The dataset spells one folder `Catterpillar` — we store it as `caterpillar` so the label
matches the knowledge base.


In [ ]:
def find_class_root(root):
    for cur, dirs, files in os.walk(root):
        subdirs = [d for d in dirs if not d.startswith('.')]
        if len(subdirs) >= 5:
            ok = 0
            for d in subdirs:
                p = os.path.join(cur, d)
                if any(is_img(f) for f in os.listdir(p)[:5]):
                    ok += 1
            if ok >= 5:
                return cur
    return root

GEN_ROOT = find_class_root(path_general)
RENAME = {'catterpillar': 'caterpillar'}
print('General class root:', GEN_ROOT)

for d in sorted(os.listdir(GEN_ROOT)):
    full = os.path.join(GEN_ROOT, d)
    if not os.path.isdir(full):
        continue
    label = RENAME.get(d.lower(), d.lower())
    files = [os.path.join(full, f) for f in os.listdir(full) if is_img(f)]
    n = copy_images(files, os.path.join(MERGED, label), f'gen_{label}')
    print(f'  {label:20s} {n:5d} images (general)')

## 5. Copy 7 crop-pest classes from IP102 (Dataset B)

IP102 stores images in numbered folders. We read IP102's own class list, then match each
target pest **by name** (not by a hard-coded number), so it works even if the numbering
differs. All of IP102's train/val/test images for a matched class are pooled together.
Each IP102 class is capped so it does not dwarf the general classes.


In [ ]:
# our label -> list of IP102 name fragments to match (lowercase, substring match)
IP102_TARGETS = {
    'brown_planthopper': ['brown plant hopper'],
    'rice_stem_borer':   ['asiatic rice borer', 'yellow rice borer', 'rice stem borer', 'striped rice borer'],
    'fall_armyworm':     ['army worm', 'armyworm'],
    'fruit_fly':         ['fruit fly', 'bactrocera', 'oriental fruit fly', 'tephritid'],
    'thrips':            ['thrips'],
    'mealybug':          ['mealybug', 'mealy bug'],
    'leafhopper':        ['rice leafhopper', 'leafhopper'],
}
CAP_PER_IP102_CLASS = 700   # keep classes roughly balanced

# 5a. Locate IP102's class-list file (id -> name) and its image folders.
classes_txt = None
for cur, _, files in os.walk(path_ip102):
    for f in files:
        if f.lower() in ('classes.txt', 'class.txt', 'classes_list.txt'):
            classes_txt = os.path.join(cur, f)
            break
    if classes_txt:
        break
print('classes.txt:', classes_txt)

id_to_name = {}
if classes_txt:
    with open(classes_txt) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                id_to_name[parts[0]] = parts[1].strip().lower()
print('IP102 classes read:', len(id_to_name))

In [ ]:
# 5b. Find IP102 split dirs that contain numbered class sub-folders.
split_dirs = []
for cur, dirs, files in os.walk(path_ip102):
    numeric = [d for d in dirs if d.isdigit()]
    if len(numeric) >= 20:  # a train/val/test folder full of class ids
        split_dirs.append(cur)
print('IP102 split dirs:', split_dirs)

# Map each target label -> matching IP102 class ids (via names).
def matches(name, fragments):
    return any(fr in name for fr in fragments)

label_to_ids = {lab: [] for lab in IP102_TARGETS}
for cid, name in id_to_name.items():
    for lab, frags in IP102_TARGETS.items():
        if matches(name, frags):
            label_to_ids[lab].append(cid)

for lab, ids in label_to_ids.items():
    names = [id_to_name[c] for c in ids]
    print(f'  {lab:20s} <- IP102 ids {ids} names {names}')

In [ ]:
# 5c. Copy the matched IP102 images into the merged folder.
for lab, ids in label_to_ids.items():
    files = []
    for cid in ids:
        for sd in split_dirs:
            cdir = os.path.join(sd, cid)
            if os.path.isdir(cdir):
                files += [os.path.join(cdir, f) for f in os.listdir(cdir) if is_img(f)]
    if not files:
        print(f'  !! {lab}: NO images found in IP102 — will be skipped. Check IP102_TARGETS names.')
        continue
    n = copy_images(files, os.path.join(MERGED, lab), f'ip_{lab}', cap=CAP_PER_IP102_CLASS)
    print(f'  {lab:20s} {n:5d} images (IP102)')

In [ ]:
# 5d. Final class summary.
classes = sorted(d for d in os.listdir(MERGED) if os.path.isdir(os.path.join(MERGED, d)))
print(f'\nTOTAL {len(classes)} classes:')
for c in classes:
    print(f'  {c:20s}', len(os.listdir(os.path.join(MERGED, c))), 'images')

## 5e. (Optional) Drop weak classes

The `thrips` class scored ~43% and was hurting `fall_armyworm`/`fruit_fly` too. Dropping the
weakest class usually lifts the overall accuracy by 2–3%. Set `DROP_CLASSES = []` to keep
everything.


In [ ]:
DROP_CLASSES = ['thrips']   # e.g. [] to keep all, or ['thrips', 'fall_armyworm'] to drop more
for c in DROP_CLASSES:
    d = os.path.join(MERGED, c)
    if os.path.isdir(d):
        shutil.rmtree(d)
        print('dropped class:', c)
    else:
        print('(not present, nothing to drop):', c)
kept = sorted(d for d in os.listdir(MERGED) if os.path.isdir(os.path.join(MERGED, d)))
print(f'\nTraining on {len(kept)} classes:', kept)

## 6. Build train / val / test datasets

80/10/10 split, images resized to 224×224. MobileNetV3 does its own preprocessing, so we
feed raw 0–255 pixels.


In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
DATA_DIR = MERGED

full_train = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='categorical')
rest = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='categorical')

class_names = full_train.class_names
print('class order:', class_names)

# Class weights: balance uneven class sizes so big classes don't dominate.
counts = {i: len(os.listdir(os.path.join(MERGED, c))) for i, c in enumerate(class_names)}
_total = sum(counts.values()); _n = len(class_names)
class_weight = {i: _total / (_n * counts[i]) for i in counts}
print('class_weight ready for', _n, 'classes')

rest_batches = tf.data.experimental.cardinality(rest).numpy()
val_ds = rest.take(rest_batches // 2)
test_ds = rest.skip(rest_batches // 2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = full_train.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 7. Data augmentation


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation')

## 8. Build the model (transfer learning)

Frozen MobileNetV3-Large base + a small classification head.


In [ ]:
NUM_CLASSES = len(class_names)
base = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 9. Train the classification head


In [ ]:
early = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True,
                                         monitor='val_accuracy')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                                 patience=3, min_lr=1e-6, verbose=1)
history = model.fit(train_ds, validation_data=val_ds, epochs=25,
                    callbacks=[early, reduce_lr], class_weight=class_weight)

## 10. Fine-tune the top of the base

Unfreeze the upper ~120 layers (more than before) and continue at a low learning rate.
We keep BatchNormalization layers frozen — unfreezing them on a small dataset makes
training unstable and hurts accuracy.


In [ ]:
base.trainable = True
for layer in base.layers[:-120]:
    layer.trainable = False
# Keep BatchNorm layers frozen for stable fine-tuning.
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=15,
                       callbacks=[early, reduce_lr], class_weight=class_weight)

## 11. Evaluate on the held-out test set


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'\nTEST ACCURACY: {test_acc*100:.2f}%  (target >= 80%)')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 10))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix — Pest Classifier')
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=90)
plt.yticks(ticks, class_names)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], ha='center', fontsize=7,
             color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True label'); plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()

## 12. Save & export


In [ ]:
model.save('pest_model.keras')
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print('Saved pest_model.keras and class_names.json')
print('Class order:', class_names)

### Optional: also export a TFLite model

Use this if your local Python cannot run TensorFlow — the backend can load `.tflite`
with the lightweight `tflite-runtime` instead.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
open('pest_model.tflite', 'wb').write(tflite_model)
print('Saved pest_model.tflite')

## 13. Download the files


In [ ]:
from google.colab import files
for f in ['pest_model.keras', 'class_names.json', 'confusion_matrix.png']:
    files.download(f)